# Riemannian Cell Flow for predicting cytokine effects on PBMCs

## Setting up the data

I begin by copying from the `100_pbmc.ipynb` notebook to get the latent space representations of the cell states produced by PCA.

In [1]:
import warnings
from pandas.errors import SettingWithCopyWarning

warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", FutureWarning)
warnings.simplefilter("ignore", SettingWithCopyWarning)

import numpy as np
import pandas as pd
import seaborn as sns
import jax
import functools
import matplotlib.pyplot as plt
import anndata as ad
import scanpy as sc
import flax.linen as nn
import optax
import cellflow
from cellflow.model import CellFlow
import cellflow.preprocessing as cfpp
from cellflow.utils import match_linear
from cellflow.plotting import plot_condition_embedding
from cellflow.preprocessing import transfer_labels, compute_wknn, centered_pca, project_pca, reconstruct_pca
from cellflow.metrics import compute_r_squared, compute_e_distance

/home/markus/university/adlr/CellFlow/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
adata = cellflow.datasets.pbmc_cytokines()

In [3]:
adata.obs.head()

,sample,species,gene_count,tscp_count,mread_count,bc1_wind,bc2_wind,bc3_wind,bc1_well,bc2_well,...,log1p_n_genes_by_counts,log1p_total_counts,total_counts_MT,pct_counts_MT,log1p_total_counts_MT,donor,cytokine,treatment,cell_type,cytokine_family
89_103_005__s1,Donor10_4-1BBL,hg38,2236,4700,8656,89,103,5,H5,p2.A7,...,7.712891,8.455530,56.0,1.191490,4.043051,Donor10,4-1BBL,cytokine,CD8 Naive,TNF
89_103_083__s1,Donor10_4-1BBL,hg38,2222,4337,8235,89,103,83,H5,p2.A7,...,7.706613,8.375169,71.0,1.637076,4.276666,Donor10,4-1BBL,cytokine,B Naive,TNF
89_103_085__s1,Donor10_4-1BBL,hg38,1690,3079,5870,89,103,85,H5,p2.A7,...,7.433075,8.032685,197.0,6.398181,5.288267,Donor10,4-1BBL,cytokine,B Intermediate/Memory,TNF
89_104_009__s1,Donor10_4-1BBL,hg38,1746,3015,5663,89,104,9,H5,p2.A8,...,7.465655,8.011686,84.0,2.786070,4.442651,Donor10,4-1BBL,cytokine,CD14 Mono,TNF
89_104_025__s1,Donor10_4-1BBL,hg38,3182,6986,13153,89,104,25,H5,p2.A8,...,8.065579,8.851807,165.0,2.361867,5.111988,Donor10,4-1BBL,cytokine,CD14 Mono,TNF


For the sake of testing the knn on a small subset, label every sample as "is_control"

In [3]:
adata.obs["condition"] = adata.obs.apply(lambda x: x["donor"] + "_" + x["cytokine"], axis=1)
adata.obs["is_control"] = adata.obs.apply(lambda x: True if x["cytokine"]=="PBS" else False, axis=1)

In [4]:
adata.write("pbmc.h5ad")

In [2]:
adata = ad.read_h5ad("pbmc.h5ad")

In [3]:
adata_small = adata[:10000,:]

In [4]:
sc.pp.normalize_total(adata_small, target_sum=1e4)
sc.pp.log1p(adata_small)

In [5]:
adata_train = adata_small[(adata_small.obs["cytokine"]!="IL-15") | (adata_small.obs["donor"]=="Donor8")].copy()
adata_test = adata_small[((adata_small.obs["cytokine"]=="IL-15") & (adata_small.obs["donor"]!="Donor8")) | (adata_small.obs["cytokine"]=="PBS")].copy()
adata_train.n_obs, adata_test.n_obs

(9886, 677)

In [10]:
adata_train.obs["is_control"]

89_103_005__s1    False
89_103_083__s1    False
89_103_085__s1    False
89_104_009__s1    False
89_104_025__s1    False
                  ...  
34_119_006__s1    False
34_119_017__s1    False
34_119_018__s1    False
34_119_036__s1    False
34_119_051__s1    False
Name: is_control, Length: 9886, dtype: bool

In [6]:
cfpp.centered_pca(adata_train, n_comps=100, keep_centered_data=False)
cfpp.project_pca(query_adata=adata_test, ref_adata=adata_train)

In [21]:
adata_train.obs.head()

,sample,species,gene_count,tscp_count,mread_count,bc1_wind,bc2_wind,bc3_wind,bc1_well,bc2_well,...,total_counts_MT,pct_counts_MT,log1p_total_counts_MT,donor,cytokine,treatment,cell_type,cytokine_family,condition,is_control
89_103_005__s1,Donor10_4-1BBL,hg38,2236,4700,8656,89,103,5,H5,p2.A7,...,56.0,1.191490,4.043051,Donor10,4-1BBL,cytokine,CD8 Naive,TNF,Donor10_4-1BBL,True
89_103_083__s1,Donor10_4-1BBL,hg38,2222,4337,8235,89,103,83,H5,p2.A7,...,71.0,1.637076,4.276666,Donor10,4-1BBL,cytokine,B Naive,TNF,Donor10_4-1BBL,True
89_103_085__s1,Donor10_4-1BBL,hg38,1690,3079,5870,89,103,85,H5,p2.A7,...,197.0,6.398181,5.288267,Donor10,4-1BBL,cytokine,B Intermediate/Memory,TNF,Donor10_4-1BBL,True
89_104_009__s1,Donor10_4-1BBL,hg38,1746,3015,5663,89,104,9,H5,p2.A8,...,84.0,2.786070,4.442651,Donor10,4-1BBL,cytokine,CD14 Mono,TNF,Donor10_4-1BBL,True
89_104_025__s1,Donor10_4-1BBL,hg38,3182,6986,13153,89,104,25,H5,p2.A8,...,165.0,2.361867,5.111988,Donor10,4-1BBL,cytokine,CD14 Mono,TNF,Donor10_4-1BBL,True


In [12]:
adata_train.X

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 1040093 stored elements and shape (9886, 2000)>

In [7]:
cf = CellFlow(adata_train, solver="otfm")

In [8]:
cf.prepare_data(
    sample_rep = "X_pca",
    control_key = "is_control",
    perturbation_covariates = {"cytokine_treatment": ("cytokine",)},
    perturbation_covariate_reps = {"cytokine_treatment": "esm2_embeddings"},
    sample_covariates = ["donor"],
    sample_covariate_reps = {"donor": "donor_embeddings"},
    split_covariates = ["donor"],
    max_combination_length = 1,
    null_value = 0.0,
)

[########################################] | 100% Completed | 101.98 ms
[########################################] | 100% Completed | 104.57 ms
[########################################] | 100% Completed | 104.08 ms


In [10]:
pca_data = cf.train_data.cell_data

## Precompute pre-metric on latent space data manifold

Now that we have our cell data in the latent space representation produced by the PCA, let us set up the knn-graph, compute the Laplacian and its first k eigenvectors to get the approximated spectral distances between the points.

In [11]:
from cellflow.preprocessing._premetric import *

# Build the kNN graph.
W = build_knn_graph(pca_data, k=30)
# Compute the normalized Laplacian
L = normalized_laplacian(W)
# Calculate the first k eigenfunctions of the Laplacian
s_dim = 10
evals, evecs = compute_eigens(L, k=s_dim)

Psi = spectral_embedding(evals, evecs, weight_type="biharmonic")

Psi_norm = np.sum(Psi**2, axis=1)

# Precompute squared distances for two known points i and j (i.e. 0 and 1):
n = Psi.shape[0]
D = np.zeros((n, n))
D = Psi_norm[:, None] + Psi_norm[None, :] - 2 * (Psi @ Psi.T)

Having computed D, we can use the saved distances to approximate the spectral embeddings of unseen points by streaming: We interpolate between the k nearest neighbors using gaussian kernel weighting.

In [13]:
cf.prepare_validation_data(
    adata_test,
    name="test",
    n_conditions_on_log_iteration=None,
    n_conditions_on_train_end=None,
)

[########################################] | 100% Completed | 101.41 ms
[########################################] | 100% Completed | 101.22 ms
[########################################] | 100% Completed | 101.64 ms


In [14]:
unseen_pca = cf.validation_data["test"].cell_data

new_point = unseen_pca[0,:]

In [15]:
nytroem = SpectralNyström(pca_data, Psi, evals, s_dim)
new_emb = nytroem.embed_point(new_point)

There is still some normalization to do. The distances between the points are rather big. The interpolation between the points does not work yet.